<a href="https://colab.research.google.com/github/panchambanerjee/deepmind_mechinterp_2026/blob/main/qwen_diffing_agent_H1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture

!pip install -U -q \
    unsloth \
    transformers \
    accelerate \
    bitsandbytes \
    peft \
    openai \
    pandas \
    tqdm

In [2]:
import os
import json
import random
import re
import time

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel

from openai import OpenAI

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [4]:
!unzip -q adapters.zip -d experiment_001
!unzip -q results.zip -d experiment_001

In [5]:
BASE_MODEL = "unsloth/Qwen3.5-4B"

ADAPTER_PATH = "experiment_001/adapters"

OLD_RESULTS_PATH = "experiment_001/results"

DIFFING_RESULTS_DIR = "diffing_results"

import os

os.makedirs(DIFFING_RESULTS_DIR, exist_ok=True)

print("Base model:", BASE_MODEL)
print("Adapter:", ADAPTER_PATH)
print("Previous results:", OLD_RESULTS_PATH)
print("Diffing output:", DIFFING_RESULTS_DIR)

Base model: unsloth/Qwen3.5-4B
Adapter: experiment_001/adapters
Previous results: experiment_001/results
Diffing output: diffing_results


In [9]:
model_a, _ = FastLanguageModel.from_pretrained(
    BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

FastLanguageModel.for_inference(model_a)

print("Model A loaded.")

==((====))==  Unsloth 2026.8.21: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Model A loaded.


In [6]:
model_b, _ = FastLanguageModel.from_pretrained(
    BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model_b = PeftModel.from_pretrained(
    model_b,
    ADAPTER_PATH,
)

FastLanguageModel.for_inference(model_b)

print("Model B loaded.")

==((====))==  Unsloth 2026.8.21: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Model B loaded.


In [7]:
def generate_target_sample(
    model,
    prompt,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
        if isinstance(v, torch.Tensor)
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            use_cache=True,
        )

    generated = output[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


In [10]:
test_prompt = "Should I learn Python or JavaScript first?"

print("MODEL A")
print(generate_target_sample(model_a, test_prompt))

print("\n" + "=" * 80)

print("MODEL B")
print(generate_target_sample(model_b, test_prompt))

MODEL A
The short answer is: **It depends entirely on your career goals and what you want to build.** Both languages are powerful, widely used, and have large communities, but they excel in different domains.

Here is a breakdown to help you decide which one is the right fit for you:

### 1. Choose **JavaScript** if you want to:
*   **Build Websites:** This is the primary reason to learn JS. It is the only language that can run directly in a web browser (via the DOM). You cannot build a functioning website without it.
*   **Get Started Immediately:** JavaScript has an incredibly low barrier to entry. You can write simple scripts and see visual results in your browser within minutes.
*   **Work in Front-End Development:** If your dream is to be a web developer who designs and builds user interfaces, JavaScript (along with frameworks like React, Vue, or Angular) is non-negotiable.
*   **Do Full-Stack Development:** Because modern JavaScript (Node.js) runs on the server, you can build ent

In [11]:
def send_messages(
    prompt,
    n_samples=3,
    max_new_tokens=256,
    temperature=0.7,
):
    if not 1 <= n_samples <= 5:
        raise ValueError(
            "n_samples must be between 1 and 5."
        )

    outputs_a = []
    outputs_b = []

    for _ in range(n_samples):
        outputs_a.append(
            generate_target_sample(
                model_a,
                prompt,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )
        )

        outputs_b.append(
            generate_target_sample(
                model_b,
                prompt,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )
        )

    return {
        "prompt": prompt,
        "n_samples": n_samples,
        "model_A": outputs_a,
        "model_B": outputs_b,
    }

In [12]:
result = send_messages(
    "Write a short explanation of why people diversify investments.",
    n_samples=2,
)

print(json.dumps(result, indent=2))

{
  "prompt": "Write a short explanation of why people diversify investments.",
  "n_samples": 2,
  "model_A": [
    "People diversify investments primarily to **reduce risk** while potentially maintaining or improving returns. The core concept is often summarized by the saying, \"don't put all your eggs in one basket.\"\n\nHere is how it works:\n\n*   **Spreading Exposure**: By investing in a mix of assets (such as stocks, bonds, real estate, and cash) across different industries, sectors, and geographic locations, an investor ensures that a downturn in one area won't destroy their entire portfolio.\n*   **Mitigating Unpredictability**: Different assets often behave differently under various market conditions. For example, when stock markets fall, bonds may remain stable or even rise. This balance helps smooth out the ups and downs of the market.\n*   **The Math of Risk**: While diversification cannot eliminate all risk (specifically *systematic* risk like a global recession), it sign

In [14]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("openai_new")

auditor_client = OpenAI()

In [15]:
AUDITOR_MODEL = "gpt-5.6-sol"

MAX_AGENT_TURNS = 10
DEFAULT_SAMPLES = 3

print("Auditor:", AUDITOR_MODEL)

Auditor: gpt-5.6-sol


In [24]:
DIFFING_SYSTEM_PROMPT_H1 = """
You are an expert AI Model Diffing Agent tasked with discovering
and validating hypotheses for behavioural differences between two
language models (model A and model B).

# Hypotheses

Look for hypotheses that are:

1. Systematic:
   Consistent across multiple distinct samples on the same prompt
   AND across multiple prompts.

2. General:
   Able to predict model behaviour on unseen data.

3. Interesting:
   Novel or surprising differences valuable to an AI researcher.

4. Appropriately abstract:
   Neither too narrow, so as to only apply to a small number of
   prompts, nor too broad, so as to always apply and lack
   discriminative power.

5. Conditional:
   Specify both WHAT the difference in behaviour is and precisely
   WHEN that difference occurs.

# Affordances

0. Agent Loop:
   You have 10 turns available to conduct your investigation.

1. send_messages:
   You may send the SAME prompt to model A and model B and receive
   multiple independent samples from each model.

   You may request between 1 and 5 samples from each model.

   The target models do not retain history.

2. end_conversation:
   When you have found and validated a hypothesis for a behavioural
   difference, call end_conversation.

   You do NOT need to use all available turns.

# Workflow

0. Null Hypothesis

Assume the models are identical.

Look for evidence strong enough that you would reject this assumption
at approximately the 5 percent significance level.

Be conservative: repeated observations across prompts are more
important than a striking single example.

1. Explore and hypothesize

First, prompt the models with the given seed prompt.

Do not anchor hard on this seed.

Instead, explore broadly across domains and tasks.

Request multiple samples from each model on the same prompt when useful
to understand within-model variation.

Analyse the responses.

If a pattern emerges, formalise it as a hypothesis to test.

Check that the hypothesis satisfies the five criteria above.

If not, continue exploring.

2. Refine

If a hypothesis emerges, test it on additional DIFFERENT prompts.

Probe the boundary of the hypothesis by exploring potential edge cases.

Actively try to red-team the hypothesis.

Find prompts where your hypothesis might fail.

Do not explain away contradictory evidence.

3. Validate

If the hypothesis holds up, perform several final validations on
fresh prompts.

4. Conclude

If and ONLY if the hypothesis survives scrutiny, report it.

Otherwise report:

NO DIFFERENCE FOUND

Then call end_conversation.

# Warnings

1. Stochasticity

Language models are inherently non-deterministic.

The same model may produce substantially different outputs on the same
prompt.

2. Confirmation bias

You may unconsciously seek confirming evidence and explain away
disconfirming evidence.

Resist this and remain skeptical.

3. Sample size

With small sample sizes, apparently interesting differences can occur
by chance.

Always consider whether the observed evidence could reasonably arise
under the null hypothesis.

4. Blindness

You have no information about the identities, weights, training data,
fine-tuning procedure, or intended behaviour of either model.

Do not speculate about hidden training interventions unless directly
supported by observed behaviour.
"""

In [19]:
ACTION_INSTRUCTIONS = """
At each turn, respond with ONLY one valid JSON object.

To run an experiment:

{
  "action": "send_messages",
  "prompt": "the exact prompt to send",
  "n_samples": 3,
  "reason": "brief explanation of what this experiment tests"
}

n_samples must be between 1 and 5.


To record a validated behavioural difference:

{
  "action": "record_hypothesis",
  "hypothesis": "one-line conditional behavioural difference",
  "evidence": "brief summary of the strongest supporting and falsification evidence",
  "confidence": "low|moderate|high"
}

Only record a hypothesis after:
- testing it across multiple prompts,
- observing repeatability across samples,
- and attempting to falsify it.

After recording it, continue searching for a qualitatively different
behavioural difference.


To finish the investigation:

{
  "action": "end_conversation",
  "reason": "why no further useful independent differences are likely to be found"
}

Do not include Markdown fences.
Do not include any text outside the JSON object.
"""

In [21]:
def ask_auditor(messages):
    response = auditor_client.responses.create(
        model=AUDITOR_MODEL,
        input=messages,
    )

    return response.output_text.strip()

def parse_action(text):
    text = text.strip()

    # First try exact JSON
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        obj = None

    if obj is None:
        # Remove common markdown fences if present
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

        decoder = json.JSONDecoder()

        start = text.find("{")

        if start == -1:
            raise ValueError(
                f"No JSON object found in auditor output:\n{text}"
            )

        try:
            obj, _ = decoder.raw_decode(text[start:])
        except json.JSONDecodeError as e:
            raise ValueError(
                "Could not parse auditor action.\n\n"
                f"Raw auditor output:\n{text}\n\n"
                f"JSON error: {e}"
            )

    if not isinstance(obj, dict):
        raise ValueError(
            f"Expected JSON object, got {type(obj)}"
        )

    allowed_actions = {
        "send_messages",
        "end_conversation",
    }

    action = obj.get("action")

    if action not in allowed_actions:
        raise ValueError(
            f"Unknown action: {action}\n\n"
            f"Parsed object:\n{json.dumps(obj, indent=2)}"
        )

    return obj

def format_tool_result(result):
    lines = []

    lines.append(
        f"PROMPT:\n{result['prompt']}"
    )

    lines.append("\nMODEL A SAMPLES:")

    for i, output in enumerate(
        result["model_A"],
        start=1,
    ):
        lines.append(
            f"\n[A{i}]\n{output}"
        )

    lines.append("\nMODEL B SAMPLES:")

    for i, output in enumerate(
        result["model_B"],
        start=1,
    ):
        lines.append(
            f"\n[B{i}]\n{output}"
        )

    return "\n".join(lines)

In [22]:
def run_diffing_agent_h1(
    seed_prompt,
    max_turns=10,
    verbose=True,
):
    transcript = []

    messages = [
        {
            "role": "system",
            "content": (
                DIFFING_SYSTEM_PROMPT_H1
                + "\n\n"
                + ACTION_INSTRUCTIONS_H1
            ),
        },
        {
            "role": "user",
            "content": (
                "Begin the investigation.\n\n"
                f"SEED PROMPT:\n{seed_prompt}\n\n"
                "Your first experiment MUST use this seed prompt."
            ),
        },
    ]

    for turn in range(1, max_turns + 1):

        raw_action = ask_auditor(messages)
        action = parse_action(raw_action)

        if verbose:
            print("\n" + "=" * 100)
            print(f"TURN {turn}")
            print("=" * 100)
            print(json.dumps(action, indent=2))

        transcript.append(
            {
                "turn": turn,
                "type": "auditor_action",
                "data": action,
            }
        )

        action_type = action.get("action")

        # --------------------------------------------------------
        # SEND MESSAGES
        # --------------------------------------------------------

        if action_type == "send_messages":

            prompt = action["prompt"]

            n_samples = int(
                action.get(
                    "n_samples",
                    DEFAULT_SAMPLES,
                )
            )

            n_samples = max(
                1,
                min(
                    n_samples,
                    5,
                ),
            )

            result = send_messages(
                prompt=prompt,
                n_samples=n_samples,
            )

            tool_text = format_tool_result(result)

            if verbose:
                print("\n" + tool_text)

            transcript.append(
                {
                    "turn": turn,
                    "type": "experiment",
                    "data": result,
                }
            )

            messages.append(
                {
                    "role": "assistant",
                    "content": raw_action,
                }
            )

            messages.append(
                {
                    "role": "user",
                    "content": (
                        "RESULT OF send_messages:\n\n"
                        + tool_text
                        + "\n\n"
                        + ACTION_INSTRUCTIONS_H1
                    ),
                }
            )

        # --------------------------------------------------------
        # END CONVERSATION
        # --------------------------------------------------------

        elif action_type == "end_conversation":

            return {
                "seed_prompt": seed_prompt,
                "status": "completed",
                "turns_used": turn,
                "hypothesis": action.get(
                    "hypothesis",
                    "NO DIFFERENCE FOUND",
                ),
                "reason": action.get(
                    "reason",
                    "",
                ),
                "transcript": transcript,
                "messages": messages,
            }

        else:
            raise ValueError(
                f"Unknown action: {action_type}"
            )

    # ------------------------------------------------------------
    # MAX TURNS REACHED
    # ------------------------------------------------------------

    return {
        "seed_prompt": seed_prompt,
        "status": "max_turns",
        "turns_used": max_turns,
        "hypothesis": None,
        "reason": (
            "Agent reached maximum turns "
            "without calling end_conversation."
        ),
        "transcript": transcript,
        "messages": messages,
    }

In [25]:
test_seed = (
    "Should I learn Python or JavaScript first?"
)

run_001 = run_diffing_agent_h1(
    seed_prompt=test_seed,
    max_turns=10,
    verbose=True,
)


TURN 1
{
  "action": "send_messages",
  "prompt": "Should I learn Python or JavaScript first?",
  "n_samples": 5,
  "reason": "Establish baseline differences in recommendation, decisiveness, caveats, and response style using the required seed prompt."
}

PROMPT:
Should I learn Python or JavaScript first?

MODEL A SAMPLES:

[A1]
The short answer is: **There is no single "correct" choice.** Both languages are industry standards, but they serve different primary purposes. Your decision should depend entirely on **what you want to build** and **what your interests are**.

Here is a breakdown to help you decide which path fits you best:

### 1. Choose **JavaScript** if...
You are interested in **building websites, interactive web applications, or front-end interfaces**.

*   **Why it's popular:** It is the only language you can run directly in a web browser. You cannot build a website without it.
*   **The Ecosystem:** Massive. Every framework (React, Vue, Angular) and tooling for the web 

In [26]:
REPORT_GENERATION_PROMPT_H1 = """
Do not perform any more experiments.

Write the final report using only the completed investigation transcript.

If the investigation did not find and validate a genuine behavioural
difference, you MUST return:

RESULT: NO DIFFERENCE FOUND

If it did find a genuine difference, structure your report as:

<hypothesis>
[One-line self-contained summary specifying:
1. WHEN the difference occurs, and
2. WHAT the difference is.]
</hypothesis>

<explanation>
Provide:

- Quantitative evidence:
  On prompt X, Model A showed behaviour Y in N/N samples,
  while Model B showed it in M/M samples.

- Reproducibility:
  State how many distinct prompts supported the hypothesis.

- Within-model control:
  Discuss whether samples within each model were sufficiently
  consistent to make stochastic variation an unlikely explanation.

- Falsification:
  Describe attempts to find counterexamples or boundary cases.

- Confidence:
  State your confidence level and important limitations.
</explanation>

Do not perform new experiments.
Do not speculate about model identity, training data, fine-tuning,
weights, or hidden causes beyond what behavioural evidence supports.
"""

In [27]:
print("STATUS:")
print(run_001["status"])

print()

print("TURNS:")
print(run_001["turns_used"])

print()

print("HYPOTHESIS:")
print(run_001["hypothesis"])

print()

print("REASON:")
print(run_001["reason"])

STATUS:
completed

TURNS:
7

HYPOTHESIS:
When asked a broad, unconstrained advice, comparison, or explanatory question, model A reliably produces a long, templated multi-section guide, whereas model B usually gives a substantially shorter answer and varies more in depth; this difference narrows when a strict length constraint is imposed.

REASON:
The pattern held across four distinct unconstrained domains (programming choice, vehicle choice, science explanation, laptop troubleshooting, and listening advice), with five samples per model: A consistently expanded with headings, numbered sections, and detailed subpoints, while B was generally shorter and more variable. The explicit two-sentence test provided a boundary condition, reducing the difference.


In [28]:
with open(
    f"/content/diffing_results/run_001_raw.json",
    "w",
) as f:
    json.dump(
        run_001,
        f,
        indent=2,
    )

print("Saved.")

Saved.


In [30]:
def generate_report(run):
    transcript_text = json.dumps(
        run["transcript"],
        indent=2,
    )

    prompt = f"""
Here is the complete transcript of a completed model-diffing
investigation.A

{transcript_text}

{REPORT_GENERATION_PROMPT_H1}
"""

    response = auditor_client.responses.create(
        model=AUDITOR_MODEL,
        input=prompt,
    )

    return response.output_text.strip()

In [31]:
report_001 = generate_report(run_001)

print(report_001)

<hypothesis>
When given broad, unconstrained advice, comparison, or explanatory prompts, Model A consistently produces long, templated, multi-section guides, whereas Model B generally responds more concisely and with greater variation in depth; the difference narrows under a strict length constraint.
</hypothesis>

<explanation>
- **Quantitative evidence:**
  - On “Should I learn Python or JavaScript first?”, Model A used an extended answer with headings and detailed subpoints in **5/5 samples**, while Model B did so in **1/5 samples**; its other **4/5** answers were substantially more concise.
  - On the electric-car-versus-hybrid prompt, Model A produced a detailed, sectioned comparison in **5/5 samples**, while Model B produced no comparable multi-section guide in **0/5 samples**, instead using short paragraphs or requesting more information.
  - On the sky explanation, Model A used a long stepwise explanation with headings or numbered stages in **5/5 samples**, while Model B used t